In [2]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [3]:
load_dotenv()

False

In [4]:

class SubState(TypedDict):

    input_text: str
    translated_text: str

In [ ]:

subgraph_llm = ChatOpenAI(model='gpt-4o')

In [ ]:
def translate_text(state: SubState):

    prompt = f"""
Translate the following text to Hindi.
Keep it natural and clear. Do not add extra content.

Text:
{state["input_text"]}
""".strip()
    
    translated_text = subgraph_llm.invoke(prompt).content

    return {'translated_text': translated_text}

In [ ]:
subGraph_builder = StateGraph(SubState)

subGraph_builder.add_node('translated_text', translate_text)

subGraph_builder.add_edge(START, 'translated_text')
subGraph_builder.add_edge('translated_text', END)

subgraph= subGraph_builder.compile()

In [5]:
class ParentState(TypedDict):

    question: str
    answer_eng: str
    answer_hin: str
    

In [ ]:
parent_llm = ChatOpenAI(model='gpt-4o-mini')

In [ ]:

def generate_answer(state: ParentState):

    answer = parent_llm.invoke(f"You are a helpful assistant. Answer clearly.\n\nQuestion: {state['question']}").content
    return {'answer_eng': answer}

In [ ]:

def translate_answer(state: ParentState):

    # call the subgraph
    result = subgraph.invoke({'input_text': state['answer_eng']})

    return {'answer_hin': result['translated_text']}

In [6]:
parent_builder = StateGraph(ParentState)

parent_builder.add_node("answer", generate_answer)
parent_builder.add_node("translate", translate_answer)

parent_builder.add_edge(START, 'answer')
parent_builder.add_edge('answer', 'translate')
parent_builder.add_edge('translate', END)


NameError: name 'generate_answer' is not defined

In [7]:
graph = parent_builder.compile()

graph

ValueError: Graph must have an entrypoint: add at least one edge from START to another node

In [ ]:

graph.invoke({'question': 'What is quantum physics'})